# Data management with DigitalHub

Assets are stored in repositories and registered in the catalog. 

This tutorial demonstrates how to create, update, download assets from inside and outside the platform.


## Project Initialization

Initialize our DigitalHub project. This creates a space where we can manage our data items, functions, workflows, and deployments.

We'll use the `username` as param to create a personal project in the shared space.


In [ ]:
import digitalhub as dh
import getpass as gt

USERNAME = gt.getuser()
project = dh.get_or_create_project(f"{USERNAME}-data-ops")
print(project.name)

## Data management

We can store and retrieve artifacts, data items and models via sdk, picking from local or remote sources.


In [ ]:
# generate a random dataset
import pandas as pd
import numpy as np  
df =  pd.DataFrame(np.random.rand(100, 4), columns=list('ABCD'))
df.to_csv("dataset.csv", index=False)

In [ ]:
# upload and register the dataset
di = project.log_dataitem(name="dataset", kind="table", source="dataset.csv")

In [ ]:
# we can also directly upload the dataset without saving it locally
di2 = project.log_dataitem(name="dataframe", kind="table", data=df)

Let's try with a remote resource.

In [ ]:
## remote 
URL = "https://opendata.comune.bologna.it/api/explore/v2.1/catalog/datasets/rilevazione-flusso-veicoli-tramite-spire-anno-2023/exports/csv?limit=50000&lang=it&timezone=Europe%2FRome&use_labels=true&delimiter=%3B"
filename = "rilevazione-flusso-veicoli-tramite-spire-anno-2023.csv"

In [ ]:
#import digitalhub as dh

di3 = project.new_dataitem(name=filename, kind="table", path=URL)

In [ ]:
di3

What about reading it back? 
We can get a file, either as full download or as temporary file.

In [ ]:
path = di.download(overwrite=True)

In [ ]:
path

Can we read it back?

In [ ]:
pdf = pd.read_csv(path, sep=";")

In [ ]:
pdf.head()

We can also directly load it as dataframe, without downloading a local copy.

In [ ]:
df = di.as_df(sep=";")

In [ ]:
df.head()

Maybe a parquet is more suitable?

In [ ]:
pq_file = "dataitem/dataset.parquet"
di.as_df().to_parquet(pq_file)

In [ ]:
pq_file

## Download and share the files

Let's download a copy of the dataset from the repository by generating a downloadable pre-authorized link (for example for sharing).


In [ ]:
di.spec.path

In [ ]:
from urllib.parse import urlparse
import boto3
from botocore.exceptions import ClientError

def create_presigned_url(key,ext_url=None,expiration=3600):
    try:
        s3_client = boto3.client("s3", endpoint_url=ext_url)        
        u = urlparse(key)
        response = s3_client.generate_presigned_url(
            "get_object",
            Params={"Bucket": u.netloc, "Key": str(u.path)[1:]},
            ExpiresIn=expiration,
        )
    except ClientError as e:
        raise Exception("error reading from s3")
        return None

    # The response contains the presigned URL
    return response

In [ ]:
create_presigned_url(di.spec.path)